In [ ]:
code = 'FUTURE_RANGE_BREAKOUT'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def future_range_breakout(bt, start_time, end_time, method, minute_check, sl, om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        end_dt_1m = end_dt + datetime.timedelta(minutes=10)
        
        entry_time = start_dt
        from_candle_close = True if method == 'CC' else False

        future_breakout_data = bt.future_data.loc[start_dt:start_dt+datetime.timedelta(minutes=minute_check)].copy()
        future_breakout_data.reset_index(inplace=True)
        
        if future_breakout_data.empty:
            return None
        
        lower_range_price = future_breakout_data['low'].min()
        upper_range_price = future_breakout_data['high'].max()

        start_dt = start_dt + datetime.timedelta(minutes=minute_check)
        future_check_data = bt.future_data.loc[start_dt:].copy()
        future_check_data.reset_index(inplace=True)

        lower_range_price, lower_range_flag, lower_range_time = bt.decay_check_by_given_data(future_check_data, decay_price=lower_range_price, from_candle_close=from_candle_close, orderside='SELL')
        upper_range_price, upper_range_flag, upper_range_time = bt.decay_check_by_given_data(future_check_data, decay_price=upper_range_price, from_candle_close=from_candle_close, orderside='BUY')
        
        lower_range_time = lower_range_time if lower_range_time else end_dt_1m
        upper_range_time = upper_range_time if upper_range_time else end_dt_1m
        
        if lower_range_time < upper_range_time:
            
            breakout_flag = 'LowerRange'
            breakout_time = lower_range_time
            ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(breakout_time, end_dt, om=om)
            if ce_scrip is None: return None
            
            sell_strike, sell_price  = ce_scrip, ce_price
            buy_strike , buy_price = pe_scrip, pe_price
            
        elif lower_range_time > upper_range_time:
            
            breakout_flag = 'UpperRange'
            breakout_time = upper_range_time
            ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(breakout_time, end_dt, om=om)
            if ce_scrip is None: return None
            
            sell_strike, sell_price = pe_scrip, pe_price
            buy_strike , buy_price = ce_scrip, ce_price
            
        else:
            return None
        
        # check sl
        sell_sl_price, sell_sl_flag, _, _, sell_sl_time, sell_pnl = bt.sl_check_single_leg(start_dt, end_dt, sell_strike, sl=sl, orderside='SELL', from_candle_close=from_candle_close)
        buy_sl_price, buy_sl_flag, _, _, buy_sl_time, buy_pnl = bt.sl_check_single_leg(start_dt, end_dt, buy_strike, sl=sl, orderside='BUY', from_candle_close=from_candle_close)
    
        total_pnl = sell_pnl + buy_pnl
        return [code, bt.index, start_time, end_time, method, minute_check, sl, om, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price, lower_range_price, upper_range_price, breakout_flag, breakout_time, sell_strike, sell_price, sell_sl_price, sell_sl_flag, sell_sl_time, buy_strike, buy_price, buy_sl_price, buy_sl_flag, buy_sl_time, sell_pnl, buy_pnl, total_pnl]
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, method, minute_check, sl, om])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            
            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_Method/P_MinuteCheck/P_SL/P_OM/Date/Day/DTE/EntryTime/Future/Future.LowerRange/Future.UpperRange/Breakout.Flag/Breakout.Time/Sell.Strike/Sell.Price/Sell.SL.Price/Sell.SL.Flag/Sell.SL.Time/Buy.Strike/Buy.Price/Buy.SL.Price/Buy.SL.Flag/Buy.SL.Time/Sell.PNL/Buy.PNL/Total.PNL').split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [future_range_breakout(bt, row.entry_time, row.exit_time, row.method, row.minute_check, row.sl, row.om) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))